# Limpieza de datos

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import requests

from mi_bici import data as ds
from mi_bici.config import RAW_DATA_DIR

pd.set_option('display.max_colwidth', None)

2026-10-18 15:00:32.064 | INFO     | mi_bici.config:<module>:11 - PROJ_ROOT path is: /home/chuy/Documents/mna/mlops/TC5061-Proyecto-Equipo11


In [2]:
def format_count_pct(mask: pd.Series) -> str:
    """Regresa el total y el porcentaje de registros marcados como True en mask"""
    return f'{mask.sum():,} ({mask.mean() * 100:.3f}%)'

In [3]:
# Carga datos de de viajes y estaciones

START = '2024-01'
END = '2026-09'

sources = ds.load_sources()
trip_paths = ds.download_trips(ds.select_trip_urls(sources, START, END))
stations = ds.read_stations(ds.download_stations(sources))

## Viajes (año_mes)

### Consideraciones para los viajes
**Fuente:** https://www.mibici.net/es/termino-y-condiciones/

- El tiempo máximo de viaje es de 8 horas.
- El horario de servicio es de 5am a 12:59am.



In [4]:
# Duración mínima de un viaje real cuando origen y destino son la misma estación
MIN_SAME_STATION_MINUTES = 1

# Duración máxima de un viaje válido.
MAX_TRIP_HOURS = 8

# Horario de servicio
SERVICE_START_HOUR = 5
SERVICE_END_HOUR = 1

In [5]:
# Viajes totales y estructura del dataset

frames = []
for year_month, path in trip_paths.items():
    month_trips = ds.read_trips(path)
    month_trips['year_month'] = year_month
    frames.append(month_trips)

trips = pd.concat(frames, ignore_index=True)
trips['year_month'] = trips['year_month'].astype('category')
n_trips = len(trips)

print(f'Viajes: {n_trips:,}')
print(f'Memoria: {trips.memory_usage(deep=True).sum() / 1e6:,.0f} MB')
trips.dtypes

Viajes: 12,540,018
Memoria: 439 MB


trip_id                    int64
start_time        datetime64[ns]
end_time          datetime64[ns]
origin_id                  Int32
destination_id             Int32
year_month              category
dtype: object

In [6]:
# Muestreo: Primeros 5 registros

trips.head()

,trip_id,start_time,end_time,origin_id,destination_id,year_month
0,31789291,2024-01-01 00:05:25,2024-01-01 00:29:37,294,271,2024-01
1,31789292,2024-01-01 00:07:23,2024-01-01 00:16:37,182,254,2024-01
2,31789293,2024-01-01 00:07:31,2024-01-01 00:16:37,182,254,2024-01
3,31789294,2024-01-01 00:07:56,2024-01-01 00:31:51,35,154,2024-01
4,31789295,2024-01-01 00:08:09,2024-01-01 00:31:51,35,154,2024-01


In [7]:
# Muestreo: Últimos 5 registros

trips.tail()

,trip_id,start_time,end_time,origin_id,destination_id,year_month
12540013,46255554,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,2026-09
12540014,46255556,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,2026-09
12540015,46255556,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,2026-09
12540016,46255557,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,2026-09
12540017,46255558,2026-09-30 23:59:48,2026-10-01 00:13:58,78,233,2026-09


In [8]:
# Identificar valores nulos

print(f'Nulos por columna: {trips.isna().sum().to_dict()}')

Nulos por columna: {'trip_id': 0, 'start_time': 0, 'end_time': 0, 'origin_id': 0, 'destination_id': 0, 'year_month': 0}


In [9]:
# Identificar valores duplicados

trip_columns = [column for column in trips.columns if column != 'year_month']

exact_duplicate = trips.duplicated(subset=trip_columns, keep='first')
repeated_id = trips['trip_id'].duplicated(keep=False)
conflicting_id = repeated_id & ~trips.duplicated(subset=trip_columns, keep=False)

print(f'Viajes idénticos: {format_count_pct(exact_duplicate)}')
print(f'Viajes con trip_id repetido: {format_count_pct(repeated_id)}')
print(f'Viajes con trip_id repetido y datos distintos: {format_count_pct(conflicting_id)}')

Viajes idénticos: 758 (0.006%)
Viajes con trip_id repetido: 1,516 (0.012%)
Viajes con trip_id repetido y datos distintos: 0 (0.000%)


In [10]:
# ¿Los duplicados están en el mismo archivo o en archivos distintos?

duplicated_rows = trips[trips.duplicated(subset=trip_columns, keep=False)]
months_per_trip = duplicated_rows.groupby('trip_id')['year_month'].nunique()

duplicated_rows.groupby('year_month', observed=True).size().rename('n_rows').to_frame()

,n_rows
year_month,
2026-09,1516


In [11]:
# ¿Cuántas copias tienen los registros duplicados?

duplicated_rows.groupby('trip_id').size().value_counts().rename_axis('copies').to_frame('n_trips')

,n_trips
copies,
2,758


In [12]:
# Confirmar que trip_id es único y no se repite entre archivos/meses

months_per_trip = trips.loc[repeated_id].groupby('trip_id')['year_month'].nunique()
cross_month_ids = months_per_trip[months_per_trip > 1].index

print(f'trip_id que aparecen en más de un archivo/mes: {len(cross_month_ids):,}')

trip_id que aparecen en más de un archivo/mes: 0


In [13]:
# Viajes en otro mes que no corresponde al archivo

start_month = trips['start_time'].dt.to_period('M').astype(str)
outside_month = start_month != trips['year_month'].astype(str)

print(f'Viajes fuera del mes del archivo: {format_count_pct(outside_month)}')

Viajes fuera del mes del archivo: 0 (0.000%)


In [14]:
# Duración de los viajes

duration_min = (trips['end_time'] - trips['start_time']).dt.total_seconds() / 60

duration_stats = duration_min.describe(
    percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]
).to_frame('minutes')
duration_stats['hours'] = duration_stats['minutes'] / 60
duration_stats.loc['count', 'hours'] = float('nan')

duration_stats.style.format('{:,.2f}', na_rep='')

,minutes,hours
count,"12,540,018.00",
mean,11.04,0.18
std,10.18,0.17
min,0.00,0.00
0.1%,0.00,0.00
1%,0.18,0.00
5%,2.17,0.04
50%,9.47,0.16
95%,24.32,0.41
99%,33.03,0.55


In [15]:
# Revisar duración de viajes fuera de rango "normal"

same_station = trips['origin_id'] == trips['destination_id']

non_positive = duration_min <= 0
short_same_station = (duration_min > 0) & (duration_min < MIN_SAME_STATION_MINUTES) & same_station
short_other_station = (duration_min > 0) & (duration_min < MIN_SAME_STATION_MINUTES) & ~same_station
too_long = duration_min > MAX_TRIP_HOURS * 60

print(f'Duración <= 0: {format_count_pct(non_positive)}')
print(f'Menos de {MIN_SAME_STATION_MINUTES} min en la misma estación: {format_count_pct(short_same_station)}')
print(f'Menos de {MIN_SAME_STATION_MINUTES} min entre estaciones distintas: {format_count_pct(short_other_station)}')
print(f'Más de {MAX_TRIP_HOURS} h: {format_count_pct(too_long)}')

Duración <= 0: 26,664 (0.213%)
Menos de 1 min en la misma estación: 340,142 (2.712%)
Menos de 1 min entre estaciones distintas: 22,390 (0.179%)
Más de 8 h: 367 (0.003%)


In [16]:
# Viajes con estaciones que no existen en el catálogo

catalog_ids = stations['station_id']

unknown_origin = trips['origin_id'].notna() & ~trips['origin_id'].isin(catalog_ids)
unknown_destination = trips['destination_id'].notna() & ~trips['destination_id'].isin(catalog_ids)
unknown_station = unknown_origin | unknown_destination

unknown_station_ids = sorted(
    set(trips.loc[unknown_origin, 'origin_id']) | set(trips.loc[unknown_destination, 'destination_id'])
)

rows = []
for station_id in unknown_station_ids:
    as_origin = trips['origin_id'] == station_id
    as_destination = trips['destination_id'] == station_id
    rows.append(
        {
            'station_id': station_id,
            'origin': as_origin.sum(),
            'destination': as_destination.sum(),
            'round_trips': (as_origin & as_destination).sum(),
            'n_trips': (as_origin | as_destination).sum(),
        }
    )

pd.DataFrame(rows).set_index('station_id')

,origin,destination,round_trips,n_trips
station_id,,,,
405,3,3,3,3
407,1957,2120,126,3951


In [17]:
# Viajes fuera del horario de servicio

start_hour = trips['start_time'].dt.hour

in_service = (start_hour >= SERVICE_START_HOUR) | (start_hour < SERVICE_END_HOUR)
outside_service = ~in_service

print(f'Viajes fuera del horario de servicio: {format_count_pct(outside_service)}')

Viajes fuera del horario de servicio: 139 (0.001%)


# Hallazgos
- No se encuentran valores nulos.
- 758 (0.006%) viajes duplicados. Todos los viajes duplciados están en el archivo 2026-09.
- 1,516 (0.012%) viajes con trip_id repetidos. Los trip_id se repiten en su mismo mes.
- La fecha de todos los viajes corresponden correctamente al archivo/mes en el que están registrados.
- 26,664 (0.213%) viajes con duración de 0 minutos.
- 125,400 (1%) viajes duran 11 segundos o menos.
- 340,142 (2.71%) viajes con duración menor a 1 minuto con la misma estación de origen y destino. Posible fallo en el anclaje.
- 22,390 (0.179%) viajes con duración menor a 1 minuto con diferentes estaciones entre origen y destino. Posible estaciones cerca.
- 3,954 (0.032%) viajes tienen estaciones que on se encuentran en el catálogo de estaciones.
  - 3,951 viajes en la estación 407
  - 3 viajes en la estación 405
- 367 (0.003%) viajes con más de 8 horas de viaje. El tiempo máximo de viaje es de 8 horas.
- 139 (0.001%) viajes fuera del horario de servicio. El horario de servicio es de 5 am a 12:59 am.
  